In [ ]:
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import json

es = Elasticsearch(["http://localhost:9200"])
index_name = 'ustawy'

if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)
    print(f"Stary index '{index_name}' został usunięty.")

mapping = {
    "mappings": {
        "properties": { 
            "title" : {"type": "text"},
            "text" : {"type": "text"},
            "id" : {"type": "text"},
            "_embedding": {"type": "dense_vector", "dims": 1024, "index": True, "similarity": "cosine"} 
        }
    }
}

es.indices.create(index=index_name, body=mapping)
print(f"Nowy indeks '{index_name}' został utworzony.")

model = SentenceTransformer('sdadas/mmlw-retrieval-roberta-large')

with open("chunks.json", "r", encoding="utf-8") as f:
    data = json.load(f)

actions = []
for doc in data:
    text_for_embedding = f"{doc['text']}"
    embedding = model.encode(text_for_embedding).tolist()

    doc["_embedding"] = embedding

    actions.append({
        "_index": index_name,
        "_source": doc
    })

success, failed = bulk(es, actions)
print(f"Zaindeksowano {success} dokumentow, bledy {failed}")

Stary index 'ustawy' został usunięty.


C:\Users\mikoo\AppData\Local\Temp\ipykernel_9468\254343823.py:24: DeprecationWarning: The 'body' parameter is deprecated for the 'create' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.indices.create(index=index_name, body=mapping)


Nowy indeks 'ustawy' został utworzony.


model.safetensors:  76%|#######5  | 661M/870M [00:00<?, ?B/s]

Zaindeksowano 309 dokumentow, bledy []


In [10]:
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch

es = Elasticsearch(["http://localhost:9200"])
index_name = 'ustawy'
model = SentenceTransformer('sdadas/mmlw-retrieval-roberta-large')

query = "Opłata, o której mowa w ust. 1, stanowi dochód"
query_embedding = model.encode(query).tolist()

result = es.search(
    index=index_name,
    body={
        "size": 3,
        "query": {
            "script_score": {
                "query": {"match_all": {}},
                "script": {
                    "source": "cosineSimilarity(params.query_vector, '_embedding') + 1.0",
                    "params": {"query_vector": query_embedding}
                }
            }
        }
    }
)

for hit in result['hits']['hits']:
    print(f"Score: {hit['_score']:.2f}")
    print(f"Text: {hit['_source'].get('text', '')[:300]}...")
    print("-" * 50)


Score: 1.82
Text: Art. 6.
1.
Złożenie wniosku o wpis do rejestru podlega opłacie w wysokości 100 zł.
2.
Zmiana danych objętych rejestrem oraz wykreślenie z rejestru nie podlegają opłacie.
3.
Opłata, o której mowa w ust. 1, stanowi dochód budżetu państwa.
4.
Opłatę, o której mowa w ust. 1, wnosi się na rachunek bankow...
--------------------------------------------------
Score: 1.80
Text: 1)
dane składającego wniosek:
a)
oznaczenie organu,
b)
w przypadku pokrzywdzonego:
-
imię (imiona) i nazwisko,
-
datę urodzenia,
-
numer PESEL, a w przypadku braku numeru PESEL - cechy dokumentu potwierdzającego tożsamość:
 nazwę i numer dokumentu oraz państwo jego wydania,
-
imię (imiona) i nazwisk...
--------------------------------------------------
Score: 1.80
Text: Art. 84.
1.
Kto bez wymaganego uprawnienia posługuje się tytułem, o którym mowa w art. 14 ust.
 1,
podlega karze grzywny.
2.
Jeżeli sprawca czynu określonego w ust. 1 działa w celu osiągnięcia korzyści majątkowej,
podlega karze grzywny 

C:\Users\mikoo\AppData\Local\Temp\ipykernel_9468\708614247.py:11: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  result = es.search(
